In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.4.6


In [2]:
# Load the same intermediate artifact used in 03_modeling.ipynb.
train_bureau = pd.read_csv('../data/processed/train_bureau.csv')
print(train_bureau.shape)

(307511, 156)


In [3]:
train_bureau.head(2)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_CREDIT_TYPE_NUNIQUE,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT,BUREAU_CREDIT_ACTIVE_CLOSED_COUNT,BUREAU_CREDIT_ACTIVE_SOLD_COUNT,BUREAU_CREDIT_TYPE_MODE_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,2.0,-499.875,-7.0,0.0,0.0,2.0,0.0,6.0,0.0,4.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,2.0,-816.000,-43.0,0.0,NaN,1.0,0.0,3.0,0.0,2.0


In [4]:
train_bureau.columns

Index(['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_CREDIT', 'AMT_ANNUITY',
       ...
       'BUREAU_CREDIT_TYPE_NUNIQUE', 'BUREAU_DAYS_CREDIT_UPDATE_MEAN',
       'BUREAU_DAYS_CREDIT_UPDATE_MAX', 'BUREAU_AMT_ANNUITY_SUM',
       'BUREAU_AMT_ANNUITY_MEAN', 'BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT',
       'BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT',
       'BUREAU_CREDIT_ACTIVE_CLOSED_COUNT', 'BUREAU_CREDIT_ACTIVE_SOLD_COUNT',
       'BUREAU_CREDIT_TYPE_MODE_COUNT'],
      dtype='str', length=156)

In [5]:
# Bin EXT_SOURCE_2 into 10 quantile-based groups (deciles), same technique
# already used in the EDA default-rate-by-decile analysis.

woe_df = pd.DataFrame({
    'EXT_SOURCE_2': train_bureau['EXT_SOURCE_2'],
    'TARGET': train_bureau['TARGET']
})
woe_df['bin'] = pd.qcut(woe_df['EXT_SOURCE_2'], q=10, duplicates='drop')

print(woe_df['bin'].value_counts().sort_index())

bin
(-0.0009999183, 0.216]    30686
(0.216, 0.34]             30685
(0.34, 0.44]              30687
(0.44, 0.512]             30684
(0.512, 0.566]            30684
(0.566, 0.608]            30687
(0.608, 0.646]            30683
(0.646, 0.682]            30694
(0.682, 0.722]            30676
(0.722, 0.855]            30685
Name: count, dtype: int64


In [6]:
# Calculate WoE and IV per bin.
# WoE = ln(% good in bin / % bad in bin), using TARGET=0 as "good" and TARGET=1 as "bad".
# IV = sum of (% good - % bad) * WoE across all bins.

total_good = (woe_df['TARGET'] == 0).sum()
total_bad = (woe_df['TARGET'] == 1).sum()

grouped = woe_df.groupby('bin', observed=True).agg(
    n_total=('TARGET', 'count'),
    n_bad=('TARGET', 'sum')
).reset_index()

grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
grouped['pct_good'] = grouped['n_good'] / total_good
grouped['pct_bad'] = grouped['n_bad'] / total_bad
grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

iv_total = grouped['iv_component'].sum()

print(grouped[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']].round(4))
print()
print("IV total:", round(iv_total, 4))

                      bin  n_total  n_bad  pct_good  pct_bad     woe  \
0  (-0.0009999183, 0.216]    30686   5631    0.0886   0.2268 -0.9397   
1           (0.216, 0.34]    30685   3706    0.0954   0.1493 -0.4474   
2            (0.34, 0.44]    30687   3056    0.0977   0.1231 -0.2307   
3           (0.44, 0.512]    30684   2566    0.0995   0.1034 -0.0384   
4          (0.512, 0.566]    30684   2278    0.1005   0.0918  0.0908   
5          (0.566, 0.608]    30687   2042    0.1013   0.0823  0.2086   
6          (0.608, 0.646]    30683   1794    0.1022   0.0723  0.3465   
7          (0.646, 0.682]    30694   1499    0.1033   0.0604  0.5367   
8          (0.682, 0.722]    30676   1289    0.1040   0.0519  0.6942   
9          (0.722, 0.855]    30685    912    0.1053   0.0367  1.0532   

   iv_component  
0        0.1299  
1        0.0241  
2        0.0058  
3        0.0001  
4        0.0008  
5        0.0040  
6        0.0104  
7        0.0230  
8        0.0361  
9        0.0722  

IV total